# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 1 — Data Loading and Exploratory Data Analysis
**University of Basrah | Computer Engineering | 2024-2025**

Loads the BRFSS 2015 dataset, performs complete EDA, and saves outputs for Notebook 2.

### Step 1 — Import Libraries

In [1]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


### Step 2 — Configuration and Output Directories
⚠️ Update `DATA_PATH` to match your local dataset location.

In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


### Step 3 — Global Settings and Helper Functions

In [3]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [ ]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


### Step 4 — Data Loading

In [5]:
section("STEP 4 — DATA LOADING")

df_raw = pd.read_csv(DATA_PATH)
TARGET = "Diabetes_binary"
FEATURES_ORIGINAL = [c for c in df_raw.columns if c != TARGET]

df_raw[TARGET] = df_raw[TARGET].astype(int)

print(f"  Dataset shape    : {df_raw.shape}")
print(f"  Target column    : {TARGET}")
print(f"  Feature columns  : {len(FEATURES_ORIGINAL)}")
print(f"\n  Class distribution:")
print(df_raw[TARGET].value_counts())
print(f"\n  Class proportions:")
print(df_raw[TARGET].value_counts(normalize=True).round(4))

n_majority = (df_raw[TARGET] == 0).sum()
n_minority = (df_raw[TARGET] == 1).sum()
imbalance_ratio = n_majority / n_minority
print(f"\n  Imbalance Ratio  : {imbalance_ratio:.2f}:1")

save_pkl(df_raw, DIRS["reports"] / "df_raw.pkl", "Raw dataframe")
df_raw.head()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 4 — DATA LOADING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Dataset shape    : (253680, 22)
  Target column    : Diabetes_binary
  Feature columns  : 21

  Class distribution:
Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64

  Class proportions:
Diabetes_binary
0    0.8607
1    0.1393
Name: proportion, dtype: float64

  Imbalance Ratio  : 6.18:1
  [SAVED] Raw dataframe → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\df_raw.pkl


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


### Step 5 — Exploratory Data Analysis

In [6]:
section("STEP 5 — EDA")

binary_cols = [c for c in FEATURES_ORIGINAL if df_raw[c].nunique() == 2]
cont_cols   = [c for c in FEATURES_ORIGINAL if df_raw[c].nunique() > 2]

print(f"  Binary features     : {len(binary_cols)}")
print(f"  Continuous features : {len(cont_cols)}")
print(f"  Missing values      : {df_raw.isnull().sum().sum()}")
print(f"  Duplicate rows      : {df_raw.duplicated().sum()}")

save_pkl(binary_cols, DIRS["reports"] / "binary_cols.pkl", "Binary cols")
save_pkl(cont_cols,   DIRS["reports"] / "cont_cols.pkl",   "Cont cols")
save_pkl(FEATURES_ORIGINAL, DIRS["reports"] / "features_original.pkl", "Features original")

df_raw.describe().round(3)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 5 — EDA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Binary features     : 14
  Continuous features : 7
  Missing values      : 0
  Duplicate rows      : 24206
  [SAVED] Binary cols → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\binary_cols.pkl
  [SAVED] Cont cols → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\cont_cols.pkl
  [SAVED] Features original → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\features_original.pkl


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,...,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000,253680.000
mean,0.139,0.429,0.424,0.963,28.382,0.443,0.041,0.094,0.757,0.634,...,0.951,0.084,2.511,3.185,4.242,0.168,0.440,8.032,5.050,6.054
std,0.346,0.495,0.494,0.190,6.609,0.497,0.197,0.292,0.429,0.482,...,0.216,0.278,1.068,7.413,8.718,0.374,0.496,3.054,0.986,2.071
min,0.000,0.000,0.000,0.000,12.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,1.000,1.000
25%,0.000,0.000,0.000,1.000,24.000,0.000,0.000,0.000,1.000,0.000,...,1.000,0.000,2.000,0.000,0.000,0.000,0.000,6.000,4.000,5.000
50%,0.000,0.000,0.000,1.000,27.000,0.000,0.000,0.000,1.000,1.000,...,1.000,0.000,2.000,0.000,0.000,0.000,0.000,8.000,5.000,7.000
75%,0.000,1.000,1.000,1.000,31.000,1.000,0.000,0.000,1.000,1.000,...,1.000,0.000,3.000,2.000,3.000,0.000,1.000,10.000,6.000,8.000
max,1.000,1.000,1.000,1.000,98.000,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,5.000,30.000,30.000,1.000,1.000,13.000,6.000,8.000


In [7]:
# 5.1 Target Distribution
counts = df_raw[TARGET].value_counts().sort_index()
pcts   = df_raw[TARGET].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bars = axes[0].bar(["No Diabetes (0)", "Diabetes (1)"],
                   counts, color=[COLOR_NEG, COLOR_POS], edgecolor="white", lw=2)
for bar, pct in zip(bars, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1000,
                 f"{bar.get_height():,}\n({pct:.1f}%)",
                 ha="center", fontweight="bold")
axes[0].set_title("Class Distribution — Absolute Counts", fontweight="bold")
axes[0].set_ylabel("Count")

wedge_kw = dict(edgecolor="white", linewidth=3)
axes[1].pie(counts, labels=["No Diabetes", "Diabetes"],
            colors=[COLOR_NEG, COLOR_POS],
            autopct="%1.2f%%", startangle=90, wedgeprops=wedge_kw)
centre = plt.Circle((0, 0), 0.55, fc="white")
axes[1].add_patch(centre)
axes[1].set_title("Class Distribution — Proportions (Donut)", fontweight="bold")

fig.suptitle("Target Variable Distribution — Diabetes_binary",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "01_target_distribution.png")
plt.show()

  [PLOT]  01_target_distribution.png


In [8]:
# 5.2 Binary Features vs Target
ncols_b = 4
nrows_b = (len(binary_cols) + ncols_b - 1) // ncols_b
fig, axes = plt.subplots(nrows_b, ncols_b,
                          figsize=(ncols_b * 4, nrows_b * 3.5))
axes = axes.flatten()
for i, col in enumerate(binary_cols):
    for cls, color in [(0, COLOR_NEG), (1, COLOR_POS)]:
        sub = df_raw[df_raw[TARGET] == cls]
        axes[i].bar([f"{col}=0", f"{col}=1"],
                    sub[col].value_counts().sort_index(),
                    color=color, alpha=0.7,
                    label=f"Diab={cls}", edgecolor="white")
    axes[i].set_title(col, fontweight="bold", fontsize=9)
    axes[i].legend(fontsize=7)
for j in range(len(binary_cols), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Binary Features vs Target", fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "02_binary_features_vs_target.png")
plt.show()

  [PLOT]  02_binary_features_vs_target.png


In [9]:
# 5.3 Continuous Feature Distributions
nrows_c = (len(cont_cols) + 2) // 3
fig, axes = plt.subplots(nrows_c, 3, figsize=(15, nrows_c * 4))
axes = axes.flatten()
for i, col in enumerate(cont_cols):
    for cls, color, label in [(0, COLOR_NEG, "No Diabetes"),
                               (1, COLOR_POS, "Diabetes")]:
        sub = df_raw[df_raw[TARGET] == cls][col]
        axes[i].hist(sub, bins=30, alpha=0.6, color=color,
                     label=label, density=True, edgecolor="white")
    axes[i].set_title(col, fontweight="bold")
    axes[i].legend(fontsize=8)
for j in range(len(cont_cols), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Continuous Features — Distribution by Class",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "03_continuous_distributions.png")
plt.show()

  [PLOT]  03_continuous_distributions.png


In [10]:
# 5.4 Correlation Heatmap
fig, ax = plt.subplots(figsize=(17, 14))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True,
            linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Heatmap (Lower Triangle)",
             fontweight="bold", fontsize=14)
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "04_correlation_heatmap.png")
plt.show()

  [PLOT]  04_correlation_heatmap.png


In [11]:
# 5.5 Point-Biserial Correlation with Target
pb_corr = {}
for col in FEATURES_ORIGINAL:
    r, p = pointbiserialr(df_raw[TARGET], df_raw[col])
    pb_corr[col] = {"correlation": r, "p_value": p}

pb_df = (pd.DataFrame(pb_corr).T
         .sort_values("correlation", key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(11, 8))
colors = [COLOR_POS if v > 0 else COLOR_NEG
          for v in pb_df["correlation"]]
ax.barh(pb_df.index[::-1], pb_df["correlation"][::-1],
        color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.set_title("Point-Biserial Correlation — Features vs Target",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Correlation Coefficient")
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "05_target_correlation.png")
plt.show()

  [PLOT]  05_target_correlation.png


In [12]:
# 5.6 Boxplots — Continuous Features
ncols = 3
nrows = (len(cont_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4.5))
axes = axes.flatten()
for i, col in enumerate(cont_cols):
    sns.boxplot(x=TARGET, y=col, data=df_raw,
                palette={"0": COLOR_NEG, "1": COLOR_POS},
                ax=axes[i])
    axes[i].set_title(col, fontweight="bold", fontsize=12)
    axes[i].set_xticklabels(["No Diab.", "Diab."])
    axes[i].set_xlabel("")
for j in range(len(cont_cols), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Continuous Features — Boxplot by Class",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eda"] / "06_boxplots_continuous.png")
plt.show()

  [PLOT]  06_boxplots_continuous.png


### Step 6 — Imbalance Analysis and Strategy

In [13]:
section("STEP 6 — IMBALANCE ANALYSIS & STRATEGY")

n0 = (df_raw[TARGET] == 0).sum()
n1 = (df_raw[TARGET] == 1).sum()
ratio = n0 / n1

print(f"""
  Class 0 (No Diabetes) : {n0:>8,}  ({n0/(n0+n1)*100:.1f}%)
  Class 1 (Diabetes)    : {n1:>8,}  ({n1/(n0+n1)*100:.1f}%)
  Imbalance Ratio       : {ratio:.2f} : 1
""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 6 — IMBALANCE ANALYSIS & STRATEGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Class 0 (No Diabetes) :  218,334  (86.1%)
  Class 1 (Diabetes)    :   35,346  (13.9%)
  Imbalance Ratio       : 6.18 : 1



In [14]:
# Imbalance Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(["Dataset"], [n0], color=COLOR_NEG,
            label=f"No Diabetes ({n0:,})", edgecolor="white")
axes[0].bar(["Dataset"], [n1], bottom=[n0],
            color=COLOR_POS, label=f"Diabetes ({n1:,})", edgecolor="white")
axes[0].set_title("Absolute Class Counts", fontweight="bold")
axes[0].legend()

wedge_kw = dict(edgecolor="white", linewidth=2)
axes[1].pie([n0, n1], labels=["No Diabetes", "Diabetes"],
            colors=[COLOR_NEG, COLOR_POS],
            autopct="%1.2f%%", startangle=90, wedgeprops=wedge_kw)
centre = plt.Circle((0, 0), 0.55, fc="white")
axes[1].add_patch(centre)
axes[1].set_title("Class Proportion (Donut)", fontweight="bold")

ratios = {"Current\nDataset": ratio, "Mild\n(3:1)": 3,
          "Moderate\n(5:1)": 5, "Severe\n(10:1)": 10}
bar_colors = [COLOR_POS if i == 0 else "#BBBBBB"
              for i in range(len(ratios))]
axes[2].bar(list(ratios.keys()), list(ratios.values()),
            color=bar_colors, edgecolor="white")
axes[2].set_title("Imbalance Ratio Comparison", fontweight="bold")
axes[2].set_ylabel("Majority : Minority Ratio")

fig.suptitle("Class Imbalance Overview", fontsize=15, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_imb"] / "01_imbalance_overview.png")
plt.show()

  [PLOT]  01_imbalance_overview.png


In [15]:
# SMOTE Preview
if SMOTE_AVAILABLE:
    from collections import Counter
    X_demo = df_raw[FEATURES_ORIGINAL].sample(10000, random_state=SEED)
    y_demo = df_raw.loc[X_demo.index, TARGET]
    sm = SMOTE(random_state=SEED)
    X_res, y_res = sm.fit_resample(X_demo, y_demo)
    print(f"  Before SMOTE: {Counter(y_demo)}")
    print(f"  After  SMOTE: {Counter(y_res)}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, counts_d, title in [
        (axes[0], Counter(y_demo),  "Before SMOTE (10k sample)"),
        (axes[1], Counter(y_res),   "After  SMOTE (balanced)")]:
        ax.bar(["No Diabetes","Diabetes"],
               [counts_d[0], counts_d[1]],
               color=[COLOR_NEG, COLOR_POS], edgecolor="white")
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel("Count")
    fig.suptitle("SMOTE — Before vs After (10k demo)",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    savefig(fig, DIRS["plots_imb"] / "02_smote_demo.png")
    plt.show()
    del X_demo, y_demo, X_res, y_res
else:
    print("  SMOTE preview skipped.")

  Before SMOTE: Counter({0: 8640, 1: 1360})
  After  SMOTE: Counter({0: 8640, 1: 8640})
  [PLOT]  02_smote_demo.png


### ✅ Notebook 1 Complete
Outputs saved to `diabetes_unbalanced_outputs/reports/`. Run Notebook 2 next.